# Anime forum sentiment · 5,000 comments per title
Collect AniList forum discussions separately from MAL reviews. CPU is enough; a GPU is not required. Actual totals may be smaller than the cap. Sentiment is predicted by VADER, not validated against human labels.

Run the cells in order. The notebook uses the pull-request branch until it is merged. Change `BRANCH` to `main` after merging.

In [ ]:
BRANCH = "design/anime-sentiment-atlas"
LIMIT = 5000
import subprocess, os
from pathlib import Path
repo = Path("/content/anime-reviews")
if not repo.exists():
    subprocess.run(["git", "clone", "--branch", BRANCH, "https://github.com/harsh-github007/Anime-Reviews-Sentimental-Analysis.git", str(repo)], check=True)
os.chdir(repo)
subprocess.run(["python", "-m", "pip", "install", "-e", "."], check=True)

## Optional: keep checkpoints in Google Drive
Enable this before collecting to resume after a Colab disconnect. Re-running the collection resumes saved checkpoints. Raw comments stay in your own Drive; only aggregate results are used by the website.

In [ ]:
USE_DRIVE = True
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    checkpoint_dir = "/content/drive/MyDrive/AnimeForum/checkpoints"
else:
    checkpoint_dir = "data/forum"
Path(checkpoint_dir).mkdir(parents=True, exist_ok=True)
# Reuse the locally collected 12,337-comment dataset uploaded to your Drive.
import zipfile, json
archive = Path("/content/drive/MyDrive/AnimeForum/forum-checkpoints.zip")
if USE_DRIVE and archive.exists() and not list(Path(checkpoint_dir).glob("*-mal.json")):
    with zipfile.ZipFile(archive) as bundle:
        for name in bundle.namelist():
            if Path(name).name != name or not name.endswith(".json"):
                raise ValueError("Unexpected checkpoint archive entry")
            target = Path(checkpoint_dir) / name
            incoming = json.loads(bundle.read(name))
            existing = json.loads(target.read_text()) if target.exists() else {}
            if len(incoming.get("comments", [])) >= len(existing.get("comments", [])):
                target.write_text(json.dumps(incoming, ensure_ascii=False))
    print("Imported Drive checkpoints; completed titles will be reused.")

print("Reading raw JSON comments from:", checkpoint_dir)
print("Checkpoint files:", len(list(Path(checkpoint_dir).glob("*.json"))))


In [ ]:
subprocess.run(["python", "-m", "animesent", "forum", "--limit", str(LIMIT), "--data", checkpoint_dir, "--out", "results/forum.json"], check=True)

## MyAnimeList coverage without an API key
The collector reads public anime-linked forum pages if no Client ID is configured. It stops on blocked or unrecognised pages and reports partial coverage. No GPU is needed. Comments are checkpointed per title. A Client ID remains optional for using the official API.

## Inspect actual counts and collection status
A title is complete when its available discussions are exhausted or the cap is reached. If errors appear, rerun the collection cell to retry. No automatic GitHub upload occurs.

In [ ]:
import json
import pandas as pd
summary = json.loads(Path("results/forum.json").read_text())
rows = [{"Anime": a["title"], "Comments": a["comments"], "Complete": a["complete"], "Error": a["error"], **a["counts"]} for a in summary["anime"]]
display(pd.DataFrame(rows))
print("Total comments:", sum(a["comments"] for a in summary["anime"]))

In [ ]:
from google.colab import files
files.download("results/forum.json")